## Business Objective

1. To understand the impact of markdowns to sales on a per SKU and per transaction/basket level;
2. To understand if markdowns would impact overall store sales;
3. To understand what products tend to respond to markdowns the most, and which ones don't; 
4. To understand cross-sell and up-sell opportunities.


<br>

## Process: 
- <a href='#1'>Look At Data First </a>
 - <a href='#1_1'>How the entire dateset is organized?</a>
 - <a href='#3'>Explore cases when Count=0</a>
 - <a href='#4'>Explore cases when Count=-1</a>
 - <a href='#1_2'>How many different behaviors we can observe within a transaction?</a>
- <a href='#2'>Cleaning Dirty Data and Finding Outliers </a>
 - <a href='#6'>Create a column indicating whether a SKU is a regular return, orphan return</a> **(Check 'Return Types.ipynb')**
 - <a href='#7'>Check Missing Values</a>
 - <a href='#8'>Explore Anomalies</a>
  - <a href='#81'>Calculate Discount Rate on a per transaction and per sku basis </a> **(Check 'Discount Rate.ipynb')**
 - <a href='#82'>Explore Outliers</a>
- <a href='#80'>Exploratory Data Analysis</a>  **(Check 'EDA.ipynb')**
- <a href='#90'>Actual Analysis </a>
 - <a href='#9'>1. What is the impact of markdowns to sales on a per transaction level? </a>
 - <a href='#110'>2. How promotion strategy impacts sales on a per store basis? </a>
 - <a href='#10'>3. If a product is discounted across different stores, how are you going to evaluate the promotion strategy?</a>
 - <a href='#11'>4. Can we find any cross-sell and up-sell opportunites?</a>
 - <a href='#12'>5. How about the time or day of promotion? Would that make a difference?</a>
 - <a href='#13'>6. Look at low end and high end products separately when evaluating promotion strategy</a>
 
 




<br>

## Look At Data First	<a name='1' />


In [ ]:
import pandas as pd
from datetime import datetime

In [ ]:
start_time = datetime.now()
data = pd.read_csv('RetailerPromotionStrategy_data.csv')
print datetime.now()-start_time

In [ ]:
data.head()

### How the entire dateset is organized? <a name='1_1' />

In [ ]:
print 'Shape of dataset: {0}'.format(data_type.shape)
print 'Number of unique stores: {0}'.format(len(data_type.StoreID.unique()))
print 'Number of unique transactions: {0}'.format(len(data_type.TransactionID.unique()))
print 'Number of unique dates: {0}'.format(len(data_type.Date.unique()))
print 'Number of unique SKUs: {0}'.format(len(data_type.SKU.unique()))

- There are total 1M rows and 7 features in the dataset. Among the data, there are 73 unique stores, 15K unique SKUs, and 200K unique transactions.

- The records are organized on a SKU level by each transaction within each store, for each day.


### Explore cases when Count=0 <a name='3' />

In [ ]:
data_count0 = data[data['Count']==0]

data_count0.head(3)

In [ ]:
data.loc[(data['SKU']=='A256')&(data['TransactionID']=='11-7270-00007-00057')]

When Count=0, it means it's a discount applied to the purchase

In [ ]:
# Why there are rows where Count=0 but Sales>0?
data.loc[(data['Count']==0)&(data['Sales']>0)].head(3)

In [ ]:
data[(data.TransactionID=='11-7270-00007-00110')&(data.SKU=='A6007')][4:]

We know from the above table: The positive sales amount is basically the credit back to the store for the discount extended to the customer earlier.

### Explore cases when Count=-1 <a href='#4'>

In [ ]:
# Count=-1
data.loc[data['Count']==-1].head()

In [ ]:
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

In [ ]:
q = '''
    select
        d1.StoreID, d1.TransactionID, d1.Line, d1.Sales, d1.Count, d1.Date, d1.SKU
    from 
        data d1
    inner join
        (select 
            TransactionID, SKU
        from
            data
        where Count=-1) as d2
    on 
        d1.TransactionID = d2.TransactionID and
        d1.SKU = d2.SKU
    '''
data2 = pysqldf(q)

In [ ]:
data2.head(10)

We can see from the table, ?count=-1? means the customer returned the item just bought or it could be a promotion strategy like buy 1 get 1 free. In this analysis, I treated it as a return.


### How many different behaviors we can observe within a transaction? <a name='1_2'>

Within a transaction, there are only 3 behaviors which we will observe:   
1. Purchase item(s) at the regular price;   
2. Purchase item(s) at a discount;   
3. Return item(s) that were purchased either within the same transaction, or from a prior transaction.


<br><br>

## Cleaning Dirty Data and Finding Outliers <a name='2'/>

### Create a column indicating whether a SKU is a regular return, orphan return<a name='6' />

Please **check 'Return Types.ipynb'** to see how I sorted out regular returns and orphan returns. And then I create a new column next to the raw dataset and indicate whether a SKU is a regular return, orphan return, regular return credit back, or orphan return credit back. This will allow us to easily manipulate the records on a row level, SKU level, or transaction level, depending on what our analysis needs are. For example, everytime, when there is an orphan return, we simply exclude them when calculating sales per transaction. If it is a regular return, we leave them be as they will cancel each other out.


In [ ]:
data_type = pd.read_csv('type.csv')
data_type.fillna('',inplace=True)

In [ ]:
data_type.shape

In [ ]:
data_type[data_type.TransactionID=='11-7270-00007-00066'][:4]

In [ ]:
data_type[data_type.TransactionID=='11-7270-00012-00154']

In [ ]:
data_type[(data_type.TransactionID=='12-7081-00015-00118')&(data_type.SKU=='A5483')]

<br><br>

### Check Missing Values <a name='7' />

In [ ]:
data_type.isnull().sum()

<br><br>

### Explore Anomalies <a name='8' />

In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
data_type.describe()

#### Scenario 1: When count is a decimal point and sales=0


In [ ]:
data_type[(data_type.Sales==0) & (data_type.Count<1) & (data_type.Count>0)]

In [ ]:
data_type.drop(data_type[(data_type.Sales==0) & (data_type.Count<1) & (data_type.Count>0)].index,inplace=True)

There are 5 rows where count is a decimal point and sales=0.    
Treatment: We can safely filter these rows out from our analysis.

#### Scenario 2: When Count=0 and Sales=0

In [ ]:
data_type[(data_type.Sales==0) & (data_type.Count==0)].head(2)

In [ ]:
data_type[(data_type.Sales==0) & (data_type.Count==0)].shape

In [ ]:
data_type.drop(data_type[(data_type.Sales==0) & (data_type.Count==0)].index,inplace=True)

There are 3223 rows where count=0 and sales=0.    
Treatment: We can safely filter these rows out from our analysis.

#### Scenario 3: Orphan Return - returns with no matching purchase within the same transaction / returned an item purchased earlier at the regular price, AND purchased the same item at discounted price

In [ ]:
data_type[(data_type.TransactionID=='11-7270-00012-00190')]

In [ ]:
data_type[(data_type.TransactionID=='12-7081-00015-00118')&(data_type.SKU=='A5483')]

In [ ]:
orphan = data_type[(data_type.Type=='Orphan_Return')|(data_type.Type=='CreditBack_OrphanReturn')]

In [ ]:
orphan.head()

In [ ]:
print data_type.shape
print orphan.shape
print len(orphan.TransactionID.unique())

In [ ]:
# Drop rows with orphan returns
data_type = pd.merge(data_type,orphan,on=['StoreID','TransactionID','Line','Sales','Count','Date','SKU','Type'],how='left',indicator=True)
data_type = data_type[data_type['_merge']=='left_only']

In [ ]:
data_type.drop('_merge',1,inplace=True)

In [ ]:
data_type.head(3)

In [ ]:
data_type.shape

There are 3651 rows exhibit orphan returns(include credit back for orphan returns) and 2813?transactions exhibit orphan return.  

Treatment: Everytime, when there is an orphan return, we simply exclude them when calculating sales per transaction. If it is a regular return, we leave them be as they will cancel each other out.

In [ ]:
data_type.to_csv('data_type.csv', sep=',',index=False)

<br><br>

### Calculate Discount Rate on a per transaction and per sku basis <a name='81' />

Given that we calculate discounts on a per transaction and per sku basis, all we need to do is to sum up the sales and discounts to calculate the discount rate. (**Check 'Discount Rate.ipynb'**)  

Since the total sales per SKU per transaction can not be negative. With that in mind, we can just filter out these rows, and the rest, just keep them there as we have no reference as to whether a steep discount rate such as 85% is valid. For now, let's assume they are.



In [ ]:
import pandas as pd

In [ ]:
data_type = pd.read_csv('data_type.csv')
data_type.fillna('',inplace=True)

In [ ]:
data_type.shape

In [ ]:
data_type.head(3)

<br><br>

### Explore Outliers <a name='82' />

In [ ]:
df = data_type[data_type.Count>0]
df['Unit Price'] = df['Sales'] / df['Count']

In [ ]:
df.describe()

In [ ]:
# Check outliers as a reference
import numpy as np
def find_outliers_tukey(x):
    q1 = np.percentile(x,25)
    q3 = np.percentile(x,75)
    iqr = q3 - q1
    floor = q1 - 1.5*iqr
    ceiling = q3 + 1.5*iqr
    outlier_indices = list(x.index[(x < floor) | (x > ceiling)])
    outlier_values = list(x[outlier_indices])
    
    return outlier_indices, outlier_values

In [ ]:
# Outliers on SKU level
np.set_printoptions(formatter={'float_kind':lambda x: "%.2f" % x})
outlier_var = ['Unit Price']
for x in outlier_var:
    tukey_indices, tukey_values = find_outliers_tukey(df[x])
    a = np.sort(tukey_values)
    print "Feature '{0}' outliers (part):{1} & {2}\n".format(x,np.unique(a)[:10],np.unique(a)[-10:])
    print "Number of outliers: {0}\n".format(len(tukey_values))

In [ ]:
data_type[data_type.TransactionID=='11-7270-00007-00223']

**Treatment:** Leave outliers there as there is no reference for us to say they are not valid, so let's just assume they are.

<br><br>

## Exploratory Data Analysis <a name='80'/>

Please check **'EDA.ipynb'**

<br><br>

## Actual Analysis <a name='90'/>

### 1. What is the impact of markdowns to sales on a per transaction level? <a name='9' />

**Metrics:** Average sales per basket on transactions with discounted items and transactions without discounted items across all stores

**Calculation Method:** 'Sum of sales / number of transactions' for ?transactions with discounted items, and transactions without discounted items across the stores. And then compare the average sales per basket. (When there is an orphan return, we simply exclude them when calculating sales per transaction)
?


In [ ]:
data_type.head()

In [ ]:
len(data_type.TransactionID.unique())

In [ ]:
# Select transactions with discounted items 
q='''
  select
      a.*
  from
      data_type a
  inner join
      (select distinct TransactionID
       from data_type
       where Count=0) b
  on
      a.TransactionID=b.TransactionID 
  '''
data50= pysqldf(q)

In [ ]:
len(data50.StoreID.unique())

In [ ]:
# Select transactions with discounted items 
q='''
  select
      a.StoreID,
      a.TransactionID,
      sum(a.Count) as Counts,
      sum(a.Sales) as Total
  from
      data_type a
  inner join
      (select distinct TransactionID
       from data_type
       where Count=0) b
  on
      a.TransactionID=b.TransactionID
  group by
      a.TransactionID
  order by
      a.TransactionID  
  '''
data51= pysqldf(q)

In [ ]:
data51.head()

In [ ]:
data51[data51.Total<=0].head()

In [ ]:
data_type[data_type.TransactionID=='11-5634-00024-00094']

In [ ]:
# Average sales per basket (there are discount items in transactions)
avg_rev_discount = data51.Total.sum() / data51.TransactionID.size

print 'Average sales per basket for those transactions include discounted items is {0}.'.format(avg_rev_discount)

In [ ]:
# Average sales per basket in different store
# avg_rev_discount_store = data50.groupby('StoreID').Total.sum() / data50.groupby('StoreID').TransactionID.size()

In [ ]:
data50.head()

In [ ]:
#Drop transactions with discounts
data52 = pd.merge(data_type,data50,on=['StoreID','TransactionID','Line','Sales','Count','Date','SKU','Type'],how='left',indicator=True)
data52 = data52[data52['_merge']=='left_only']
data52.drop('_merge',1,inplace=True)

In [ ]:
print data_type.shape
print data50.shape
print data52.shape
print len(data51.TransactionID.unique())
print len(data52.TransactionID.unique())

In [ ]:
# Select transactions where there no discounted items 
q='''
  select
      StoreID,
      TransactionID,
      sum(Count) as Counts,
      sum(Sales) as Total
  from
      data52 
  group by
      TransactionID
  order by
      TransactionID  
  '''
data53= pysqldf(q)

In [ ]:
data53.head(3)

In [ ]:
# Average sales per basket (there aren't any discount items in transactions)
avg_rev_discount1 = data53.Total.sum() / data53.TransactionID.size

print 'Average sales per basket for those transactions include non-discounted items is {0}.'.format(avg_rev_discount1)

In [ ]:
# Average sales per basket without discount items in different store
# avg_rev_discount_store1 = data52.groupby('StoreID').Total.sum() / data52.groupby('StoreID').TransactionID.size()

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
x=np.arange(2)
width=0.05
rects1 = ax.bar(x,[avg_rev_discount1,avg_rev_discount])
ax.set_ylabel('Average Revenue Per Basket')
ax.set_ylim(0,20000)
ax.set_title('Will Promotions Drive Up Average Revenue Per Basket?')
ax.set_xticks(x+width/2 )
ax.set_xticklabels(('without discount','with discount'))
#annotate
def autolabel(rects):
    for rect in rects:
        height=rect.get_height()
        ax.text(rect.get_x()+rect.get_width()/2, 1.035*height, '{:.2f}$'.format(height), ha='center', va='bottom')
autolabel(rects1)

According to the above plot, we can conclude that promotions will drive up average sales per basket.

<br><br>

### 2. How promotion strategy impacts sales on a per store basis? <a name='110' />

**Metrics:** Average sales per basket on transactions with discounted items and transactions without discounted items for SPECIFIC STORE

**Calculation Method:** For specific store, calculate 'Sum of sales / number of transactions' for transactions with discounted items, and transactions where there are no discounted. (when there is an orphan return, we simply exclude them when calculating sales per transaction)


In [ ]:
# Select transactions with discounted items for specific store
q='''
  select
      StoreID,
      sum(Sales) as Total,
      count(distinct TransactionID) as Number_of_Transactions,
      sum(Sales)/count(distinct TransactionID) as 'Average sales per basket (with discounts)'
  from
      data50
  group by
      StoreID
  
  '''
data70= pysqldf(q)

In [ ]:
# Select transactions with discounted items for specific store
q='''
  select
      StoreID,
      sum(Sales) as Total,
      count(distinct TransactionID) as Number_of_Transactions,
      sum(Sales)/count(distinct TransactionID) as 'Average sales per basket (without discounts)'
  from
      data52
  group by
      StoreID
  
  '''
data71= pysqldf(q)

In [ ]:
data70.head(3)

In [ ]:
data71.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
x=np.arange(10)
width=0.05
rects1 = ax.bar(x,data70['Average sales per basket (with discounts)'][:10])
rects2 = ax.bar(x,data71['Average sales per basket (without discounts)'][:10])
ax.set_ylabel('Average Revenue Per Basket')
ax.set_xlabel('StoreID')
ax.set_ylim(0,23000)
ax.set_title('Will Promotions Drive Up Average Revenue Per Basket on a Per Store Basis?')
ax.set_xticks(x+width/2)
ax.set_xticklabels(('A01','A02','A03','A05','A06','A07','A08','A09','A10','A11'))
ax.legend((rects1[0],rects2[0]),('With Discounts','Without Discounts'))

According to the above plot, promotions will drive up average revenue per basket on a per store basis.

<br><br>

### 3. If a product is discounted across different stores, how are you going to evaluate the promotion strategy?
 <a name='10' />


**(a)** 

**Metrics:** Elasticity of Demand for specific SKU across stores

**Calculation Method:** Percent change in demand for specific SKU before and after discount / Percent change in price for specific SKU before and after discount (When there is an orphan return, we simply exclude them)

**Explanation:** I would compare when a SKU has regular price and when it is discounted, and observe the price elasticity. Then select top 10 SKUs based on elasticity of demand across stores which tend to respond the markdowns the most.


In [ ]:
data_type.head()

#### Example

In [ ]:
# There are discounts only on 3/11/12 for the item A2341 
data_type[(data_type['StoreID']=='A29')&(data_type['SKU']=='A2341')].head(4)

In [ ]:
data_type[(data_type['StoreID']=='A29')&(data_type['SKU']=='A2341')&(data_type['Date']=='3/12/12 00:00')].head(2)

In [ ]:
print len(data_type[(data_type['StoreID']=='A29')&(data_type['SKU']=='A2341')&(data_type['Date']=='3/11/12 00:00')&(data_type['Count']==0)])
print len(data_type[(data_type['StoreID']=='A29')&(data_type['SKU']=='A2341')&(data_type['Date']=='3/12/12 00:00')&(data_type['Count']==1)])
print len(data_type[(data_type['StoreID']=='A29')&(data_type['SKU']=='A2341')&(data_type['Date']=='3/12/12 00:00')&(data_type['Count']==0)])

For SKU A2341 in Store A29:  
**Price** on 3/11/12 is 500\$/item, price on 3/12/12 is 1100\$/item, percent change in price: 600/500=120%; **Demand** on 3/11/12 is 81 items, demand on 3/12/12 is 39 items, percent change in demand: 42/81=51.85%; **Elasticity of Demand** is: 51.85% / 120% = 0.43, which means that item A2341 in store A29 is not so elastic.

#### Calculation for each SKU across stores

In [ ]:
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

In [ ]:
# Select transactions with discounted items for specific SKU
q = '''
    select
        a.SKU, sum(b.Total_Sales+a.Total_Sales)/sum(a.Total_Count) as Unit_Price, sum(a.Total_Count) as Total_Count, 'Discount' as Type
    from
    (select StoreID, TransactionID, SKU, sum(Sales) as Total_Sales, sum(Count) as Total_Count from data_type where Count!=0
    group by TransactionID, SKU) a
    join
    (select StoreID, TransactionID, SKU, sum(Sales) as Total_Sales, sum(Count) as Total_Count from data_type where Count=0 
     group by TransactionID, SKU) b
    on
    a.TransactionID=b.TransactionID and a.SKU=b.SKU
    group by 
    a.SKU
    '''
data60 = pysqldf(q)

In [ ]:
data60.head()

In [ ]:
data60 = data60[data60.Total_Count>50]

In [ ]:
print len(data60.SKU.unique())

In [ ]:
q = '''
    select
        a.*
    from
       data_type a
    join 
        data60 b
    on
        a.SKU=b.SKU
    '''
data61 = pysqldf(q)

In [ ]:
print len(data61.SKU.unique())

In [ ]:
#Select transactions without discounted items for specific SKU

q = '''
    select
        a.SKU, sum(a.Total_Sales)/sum(a.Total_Count) as Unit_Price, sum(a.Total_Count) as Total_Count, 'Regular' as Type
    from
    (select StoreID, TransactionID, SKU, sum(Sales) as Total_Sales, sum(Count) as Total_Count from data61 where Count!=0
    group by TransactionID, SKU) a
    left join
    (select StoreID, TransactionID, SKU, sum(Sales) as Total_Sales, sum(Count) as Total_Count from data61 where Count=0 
     group by TransactionID, SKU) b
    on
        a.TransactionID=b.TransactionID and a.SKU=b.SKU
    where
        b.TransactionID is null
    group by 
        a.SKU
    '''
data62 = pysqldf(q)

In [ ]:
data62.head()

In [ ]:
data62 = data62[data62.Total_Count>50]

In [ ]:
print len(data62.SKU.unique())

In [ ]:
q = '''
    select
        a.SKU,
       (abs(a.Total_Count-b.Total_Count)/b.Total_Count) /(abs(a.Unit_Price-b.Unit_Price)/b.Unit_Price) as Elasticity
    from
        data60 a 
    join
        data62 b
    on
       a.SKU = b.SKU 
    '''
data63 = pysqldf(q)

In [ ]:
data63 = data63.sort_values(['Elasticity'], ascending=[False])

In [ ]:
data63[:10]

In [ ]:
print len(data63.SKU.unique())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def autolabel(ax):
    for rect in ax.patches:
        height=rect.get_height()
        ax.text(rect.get_x()+rect.get_width()/2, 1.05*height, '{:.2f}'.format(height), ha='center', va='bottom')

In [ ]:
plt.figure(figsize=(10,6))
ax = sns.barplot(x='SKU',y='Elasticity',data=data63[:10])
plt.ylabel('Elasticity of Demand')
plt.ylim(0,200)
plt.title('Top 10 SKUs which tend to respond the markdowns the most')
autolabel(ax)

**These 10 SKUs' elasticity of demand are far bigger than 1, which means they tend to respond the markdowns the most.**

<br><br>

**(b)** 

**Metrics:** Proportion of # discounted items bought on SKU level

**Calculation Method:**  # discounted items bought for specific SKU / total  # items bought for specific SKU (When there is an orphan return, we simply exclude them)

**Explanation:**  We want to know whether decrease in sales will drive up increas in demand for specific SKU


In [ ]:
q = '''
    select
        a.SKU,
        a.Total_Count/(a.Total_Count+b.Total_Count) as 'Proportion of the discounted items'
    from
        data60 a 
    join
        data62 b
    on
       a.SKU = b.SKU 
    '''
data64 = pysqldf(q)

In [ ]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)
data64.head()

In [ ]:
data64.describe()

In [ ]:
data64 = data64.sort_values(['Proportion of the discounted items'], ascending=[False])

In [ ]:
plt.figure(figsize=(10,6))
ax = sns.barplot(x='SKU',y='Proportion of the discounted items',data=data64[:10])
plt.ylabel('Proportion of # discounted items bought')
plt.ylim(0,1.2)
plt.title('10 SKUs which have the highest proportion of  # discounted items bought')
autolabel(ax)

<br><br>

### 4. Can we find any cross-sell and up-sell opportunites? <a name='11' />

**Metrics:** support, confidence, lift 

**Calculation Method:** For specific store, we can use association anlysis to find some relationships between 2 products, and use support, confidence, lift to know  the degree to which the antecedants' occurences are dependent on consequents' occurences. (Make sure to filter out orphan returns from the association analysis.) (There is no need to filter out the regular returns. Since when calculating sum of counts, it will cancel each other out.)


In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [ ]:
data_type = pd.read_csv('data_type.csv')
data_type.fillna('',inplace=True)

In [ ]:
data_type[(data_type.TransactionID=='11-7270-00007-00089')&(data_type.SKU=='A21166')]

In [ ]:
# Explore StoreID = A29
basket = data_type[data_type['StoreID']=='A29'].groupby(['TransactionID','SKU'])['Count'].sum().unstack().reset_index().fillna(0).set_index('TransactionID')

In [ ]:
data_type[data_type['StoreID']=='A29'].groupby(['TransactionID','SKU'])['Count'].sum().head(10)

In [ ]:
basket.head()

In [ ]:
# Number of Transactions and Products
basket.shape

In [ ]:
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

basket_sets = basket.applymap(encode_units)

In [ ]:
frequent_itemsets = apriori(basket_sets, min_support=0.005, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules.head()

#### For store=A29, rule: {A2052}-->{A25185}, 
- 1.2% support means 1.2% proportion of transactions in the dataset contain both A2052 and A25815;   
- 51.9% confidence means 51.9% who buy A2052 will also buy A25185.   
- Since lift=26.783, which lets us know the degree to which the antecedants' occurences are dependent on consequents' occurences. 

#### So we conclude item A25185 and A2052 have strong relationship.

<br><br>

### 5. How about the time or day of promotion? Would that make a difference? <a name='12' />


**Metrics:** Average sales per basket on transactions with discounted items and transactions without discounted items for SPECIFIC DATE

**Calculation Method:** 'Sum of sales / number of transactions' for ?transactions with discounted items, and transactions without discounted items for SPECIFIC DATE. And then compare the average sales per basket. Also check whether promotions on different dates would make a difference. (When there is an orphan return, we simply exclude them when calculating sales per transaction)


In [ ]:
data50.head(3)

In [ ]:
# Select transactions with discounted items for specific store
q='''
  select
      Date,
      sum(Sales) as Total,
      count(distinct TransactionID) as Number_of_Transactions,
      sum(Sales)/count(distinct TransactionID) as 'Average sales per basket (with discounts)'
  from
      data50
  group by
      Date
  
  '''
data80= pysqldf(q)

In [ ]:
data80

In [ ]:
# Select transactions with discounted items for specific store
q='''
  select
      Date,
      sum(Sales) as Total,
      count(distinct TransactionID) as Number_of_Transactions,
      sum(Sales)/count(distinct TransactionID) as 'Average sales per basket (without discounts)'
  from
      data52
  group by
      Date
  '''
data81= pysqldf(q)

In [ ]:
data81

In [ ]:
np.arange(2)

In [ ]:
def autolabel(rects):
    for rect in rects:
        height=rect.get_height()
        ax.text(rect.get_x()+rect.get_width()/2, 1.05*height, '{:.2f}'.format(height), ha='center', va='bottom')

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
#x=np.arange(2)
x=np.array([0,2])
width=0.05
rects1 = ax.bar(x,data80['Average sales per basket (with discounts)'])
rects2 = ax.bar(x+0.8,data81['Average sales per basket (without discounts)'])
ax.set_ylabel('Average Revenue Per Basket')
ax.set_xlabel('Date')
ax.set_ylim(0,20000)
ax.set_title('Would that make a difference for promotions on different date?')
ax.set_xticks(x+0.4)
ax.set_xticklabels(('3/11/12','3/12/12'))
ax.legend((rects1[0],rects2[0]),('With Discounts','Without Discounts'))
autolabel(rects1)
autolabel(rects2)

From the plot, the average revenue per basket for transactions with promotions is 82% higher than transactions without promotions on 3/11/12. And  the average revenue per basket for transactions with promotions is 100% higher than transactions without promotions on 3/12/12.

<br><br>

### 6. Look at low end and high end products separately when evaluating promotion strategy <a name='13' />


We define high end products are those whose regular unit price is bigger than \$5000. And low end products are those whose regular unit price is lower than $1000.

**Metrics:** Elasticity of Demand for specific SKU across stores

**Calculation Method:** Percent change in demand for specific SKU before and after discount / Percent change in price for specific SKU before and after discount

**Explanation:** I would make my own buckets and define which SKUs are high-end products, and which SKUs are low-end products. Then I would look at low end and high end produtcs separetely by calculating elasticity of demand for each SKU across stores.


In [ ]:
# Select high end products
q='''
  select distinct SKU
  from data_type
  where Count>0 and Sales/Count>5000
  '''
data90= pysqldf(q)

In [ ]:
data_type[data_type.SKU=='A22173'].head(3)

In [ ]:
#SKUs with elasticity
data63.head()

In [ ]:
q='''
  select b.*
  from data90 a
  join data63 b
  on a.SKU=b.SKU
  '''
data91= pysqldf(q)

In [ ]:
data91

In [ ]:
def autolabel(ax):
    for rect in ax.patches:
        height=rect.get_height()
        ax.text(rect.get_x()+rect.get_width()/2, 1.05*height, '{:.2f}'.format(height), ha='center', va='bottom')

In [ ]:
import seaborn as sns
plt.figure(figsize=(10,6))
ax = sns.barplot(x='SKU',y='Elasticity',data=data91)
plt.ylabel('Elasticity of Demand')
plt.ylim(0,60)
plt.title('High-End Products')
autolabel(ax)

For high-end products, the elasticity of demand ranged from 5.8 to 38.47, which means all of them are elastic.

<br>

In [ ]:
# Select low end products
q='''
  select distinct SKU
  from data_type
  where Count>0 and Sales/Count<1000
  '''
data92= pysqldf(q)

In [ ]:
q='''
  select b.*
  from data92 a
  join data63 b
  on a.SKU=b.SKU
  '''
data93= pysqldf(q)

In [ ]:
print data91.shape #high-end products
print data93.shape #low-end products

In [ ]:
import seaborn as sns
plt.figure(figsize=(10,6))
ax = sns.barplot(x='SKU',y='Elasticity',data=data93[:10])
plt.ylabel('Elasticity of Demand')
plt.ylim(0,200)
plt.title('Low-End Products')
autolabel(ax)

In [ ]:
data93.describe()

For low-end products, the elasticity of demand ranged from 0.067 to 170.87, which means some products are inelastic and some are elastic.